In [0]:
data=spark.read.format('csv').option('inferSchema',True).option('header',True).load("/Volumes/sample_data/sales/bigmart/BigMart Sales.csv")

In [0]:
data.printSchema()

In [0]:
dbutils.fs.ls("/Volumes/sample_data/sales/bigmart")
data.show()
data.count()
data.columns
data.describe().show()
data.describe("Item_Outlet_Sales")

In [0]:
data.display()

In [0]:
df_json=spark.read.format('json').option('inferSchema',True).option('header',True).option('multiLine',False).load('/Volumes/sample_data/sales/driver/drivers.json')

In [0]:
df_json.display()

## DDL: Data Definition Language

### Schema defined with StructType() and StructField()

In [0]:
## Easy way to edit a schema - copy the original schema and edit the column type.
my_ddl_schema = ''' 
                Item_Identifier STRING, 
                Item_Weight STRING, 
                Item_Fat_Content STRING, 
                Item_Visibility DOUBLE, 
                Item_Type STRING, 
                Item_MRP DOUBLE, 
                Outlet_Identifier STRING, 
                Outlet_Establishment_Year INT, 
                Outlet_Size STRING, 
                Outlet_Location_Type STRING, 
                Outlet_Type STRING, 
                Item_Outlet_Sales DOUBLE
                '''

In [0]:
data_new = spark.read.format('csv') \
                .schema(my_ddl_schema) \
                .option('header',True)\
                .load("/Volumes/sample_data/sales/bigmart/BigMart Sales.csv")


In [0]:
data_new.printSchema()

### StructType() Schema


In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
my_struct_schema = StructType([
    StructField('Item_Identifier',StringType(),nullable=False),
    StructField('Item_Weight',DoubleType(), nullable=True),
    StructField('Item_Fat_Content',StringType(),nullable=True),
    StructField('Item_Visibility',DoubleType(), nullable=True),
    StructField('Item_MRP',DoubleType(), nullable=True),
    StructField('Outlet_Identifier',StringType(), nullable=False),
    StructField('Outlet_Establishment_Year',IntegerType(), nullable=True),
    StructField('Outlet_Size',StringType(),nullable=True),
    StructField('Outlet_Location_Type',StringType(),nullable=True),
    StructField('Outlet_Type',StringType(),nullable=True),
    StructField('Item_Outlet_Sales',DoubleType(), nullable=True)
    ])

## SELECT and ALIAS

In [0]:
aliases ={
    'Item_Identifier':'item_id',
    'Item_Weight':'item_weight',
    'Item_Fat_Content':'fat_content',
    'Item_Visibility':'visibility',
    'Item_Type': 'item_type',
    'Item_MRP':'mrp',
    'Outlet_Identifier':'outlet_id',
    'Outlet_Establishment_Year':'outlet_est_year',
    'Outlet_Size':'outlet_size',
    'Outlet_Location_Type':'outlet_loc_type',
    'Outlet_Type':'outlet_type',
    'Item_Outlet_Sales':'sales'
}

df_main = data.select([data[col].alias(aliases[col]) for col in data.columns])
df_main.display()


## FILTER

In [0]:
""" 
Filter the data with fat content = Regular
Slice the data with item_type=Soft drink and weight<10
Fetch data with Tier in (Tier 1 or Tier 2) amd outliet size is null
"""
## Scenario 1
df_main.filter(col('fat_content')=='Regular').display()


In [0]:
## Scenario
df_main.filter((col('item_type')=='Soft Drinks') & (col('item_weight')<10)).display()

In [0]:
## Scenario 3
df_main.filter((col('outlet_size').isNull()) & (col('outlet_loc_type').isin(['Tier 1','Tier 2']))).display()

## Rename Column

In [0]:
example = data.withColumnRenamed("Item_Weight","item_wt").display()

## withColumn

In [0]:
## Create a new column
from pyspark.sql.functions import *
"""lit() let you put a fixed value into a dataframe expression
withColumn() allows you to create new column and modify existing column(s)
"""

df = data.withColumn('flag', lit('new'))
df.display()

In [0]:
df.withColumn('multiplier', col("Item_Weight") * col("Item_MRP")).display()

In [0]:
## Modify existing column
df.withColumn('Item_Fat_Content',regexp_replace(col("Item_Fat_Content"), "Low Fat", "LF")) \
    .withColumn('Item_Fat_Content',regexp_replace(col("Item_Fat_Content"), "Regular", "Reg")).display()

## Type Casting

In [0]:
df_test = df.withColumn('Item_Weight', col('Item_Weight').cast(StringType()))
df_test.printSchema()

## Sort and orderBy

In [0]:
df.sort(col('Item_Weight').desc()).display()

In [0]:
df.sort(['Item_Weight','Item_Visibility'],ascending=[False,True]).display()

## Limit

## Drop

In [0]:
df.drop('flag','multiplier').display()

## Drop Duplicates

In [0]:
df_drop = df.dropDuplicates().display()

## Union and Union ByName

### Union()

* The union() method combines DataFrames by pairing columns based on their index order, **completely ignoring column names**

* Best Used For: DataFrames that share the exact same schema in the exact same order

* The Risk: If your schemas differ in order, PySpark will happily misalign your data into the wrong columns without throwing an error, as long as the data types are compatible


In [0]:
df1 = spark.createDataFrame([(1, "Alice")], ["id", "name"])
df2 = spark.createDataFrame([(2, "Bob")], ["emp_id", "first_name"])
df1.union(df2).show()

### unionByName()
* Introduced to prevent positional alignment bug.

* unionByName() matches columns using their string names, rearranging columns order under the hood to ensure they merge correctly

* Best Used For: DataFrames that have the same columns but in a different order, or DataFrames with slightly different schemas.

In [0]:
df3 = spark.createDataFrame([(1, "Kim")], ["id", "name"])
df4 = spark.createDataFrame([("Tom", 2)], ["name", "id"])

df3.unionByName(df4).show()

### Handling missing columns (allowMissingColumns)

* By default, unionByName() fails if one DataFrame has columns the other lacks. Setting allowMissingColumns=True will bypass this, padding the missing values with null

In [0]:
df5 = spark.createDataFrame([(1, "Anne")], ["id", "name"])
df6 = spark.createDataFrame([(2, "Rob", "HR")], ["id", "name", "dept"])

df5.unionByName(df6, allowMissingColumns=True).show()

### Pro Tips - Combining multiple dataframes

In [0]:
from functools import reduce 
from pyspark.sql import DataFrame 

dfs = [df1,df2,df3,df4,df5,df6]
combined_df = reduce(lambda df_a, df_b: df_a.unionByName(df_b, allowMissingColumns=True), dfs)

In [0]:
combined_df.display()

## String Functions - upper(), lower(), initcap()


In [0]:
from pyspark.sql.functions import * 

text = [("spark SQL",), ("hello WORLD",), ("aB cD",)]

text_df = spark.createDataFrame(text, ["mixed_text"])

text_df.display()


In [0]:
corrected_df = text_df.withColumn("proper_case", initcap(col("mixed_text")))
corrected_df.display()

## Date Functions - current_date(), date_add(), date_sub()

### Current date and Timestamp

In [0]:
data = data.withColumn("curr_date", current_date()).withColumn("curr_time", current_timestamp())
data.display()

### Formatting and Parsing

* date_format(date, format): Converts a date/timestamp to a string based on a specific pattern (e.g., "dd/MM/yyyy").
* to_date(col, [format]): Casts a string or timestamp to a clean DateType. An optional format pattern can parse non-standard strings.
* to_timestamp(col, [format]): Casts a string or date to a TimestampType

In [0]:
from pyspark.sql.functions import date_format, to_date, to_timestamp

ex1 = [("15-08-2026", "2026-12-25 10:30:00")]

df_ex1 = spark.createDataFrame(ex1, ["str_date","str_time"])

df_ex1.select(
    to_date(col("str_date"),"dd-MM-yyyy").alias("parsed_date"),
    to_timestamp(col("str_time")).alias("parsed_time"),
    date_format(to_date(col("str_date"),"dd-MM-yyyy"), "MMMM dd,yyyy").alias("fancy_string")
).show()


### Date Arithmethics and Differences

* date_add(date, days): Adds a specific number of days to a date.
* date_sub(date, days): Subtracts a specific number of days from a date.
* datediff(end, start): Returns the difference between two dates in days (end - start).
* add_months(date, months): Adds/subtracts a specific number of months.
* months_between(end, start, [roundOff]): Returns the difference between two dates in months (as a float).


In [0]:
from pyspark.sql.functions import date_add, date_sub, datediff, add_months, months_between

ex2 = [("2026-01-01", "2026-06-01")]
df_ex2 = spark.createDataFrame(ex2, ["d1", "d2"])

df_ex2.select(
    date_add("d1",5).alias("add_days"),
    date_sub("d2",5).alias("sub_days"),
    datediff("d2","d1").alias("dates_diff"),
    add_months("d1",3).alias("add_months"),
    months_between("d2","d1").alias("months_between")

).show()

### Part Extraction

* year(col), quarter(col), month(col), dayofmonth(col): Extract numerical calendar chunks.
* dayofweek(col): Returns an integer from 1 (Sunday) to 7 (Saturday).
* dayofyear(col): Returns the day integer within the year (1–366).
* weekofyear(col): Returns the week integer within the year (1–53).
* hour(col), minute(col), second(col): Isolate specific time blocks from a timestamp.


In [0]:
from pyspark.sql.functions import year, month, dayofmonth, dayofweek, hour

df_ex3 = spark.createDataFrame([("2026-10-10 14:15:30")], ["ts"])

df_ex3.select(
    year("ts").alias("year"),
    month("ts").alias("month"),
    dayofmonth("ts").alias("day"),
    dayofweek("ts").alias("day_of_week"),
    hour("ts").alias("hour")
).show()

### Period alignment and Rounding

* trunc(date, format): Truncates a date to the unit specified by the format string ("year"/"yyyy" or "month"/"mm").
* date_trunc(format, timestamp): Similar to trunc, but acts on timestamps and accepts down to "hour", "minute", or "second".
* next_day(date, dayOfWeek): Returns the first date which is later than the value of the date column and matches the explicitly named day of the week (e.g., "Mon", "Sunday").
* last_day(date): Returns the last day of the month that the given date belongs to.


In [0]:
from pyspark.sql.functions import trunc, date_trunc, last_day, next_day

df_ex4 = spark.createDataFrame([("2026-10-12 14:45:00",)], ["ts"])

df_ex4.select(
    trunc("ts","year").alias("year_trunc"),
    date_trunc("hour", "ts").alias("hour_trunc"),
    next_day("ts","Friday").alias("next_day"),
    last_day("ts").alias("last_day")
).show()

### UNIX Epoch Conversion
Great for API integrations or processing raw system log metrics.
* from_unixtime(col, [format]): Converts the number of seconds from the UNIX epoch (1970-01-01 00:00:00 UTC) into a string format.
* unix_timestamp([col], [format]): Converts a time string or date to a UNIX timestamp integer (in seconds).

In [0]:
from pyspark.sql.functions import from_unixtime, unix_timestamp

df_ex5 = spark.createDataFrame([("2026-10-10 00:00:00", 1791590400)], ["ts_str", "epoch_int"])

df_ex5.select(
    unix_timestamp("ts_str").alias("to_epoch"),
    from_unixtime("epoch_int").alias("to_ts_string")
).show()

In [0]:
data.printSchema()

In [0]:
from pyspark.sql.functions import split, col
data_type = data.withColumn('Type', split(col('Outlet_Type'), ' ').getItem(1))

In [0]:
data_type.display()

**Note**: In case you want to expand every elemnent in its own row, use **explode()**

- Sitll need to split the column but you don't need to use getItem()

data_exploade = data.select('Outlet_Type', explode(col('Type')).alias('Type'))

### Array_Contains

- Check if an array contain specific values

In [0]:
from pyspark.sql import functions as F

data_sample = [
    ("Alice", ["Python", "SQL", "Java"]),
    ("Bob", ["Java", "C++"]),
    ("Charlie", None),
    ("David", ["Python"])
]

df_ex6 = spark.createDataFrame(data_sample, ["name", "languages"])

df_filterd = df_ex6.filter(F.array_contains("languages", "Python"))

df_filterd.show()

In [0]:
data_split = data.withColumn("outlet_type", split('Outlet_Type', ' '))
data_split.display()



In [0]:
data_with_check = data_split.withColumn('Type1_flag', F.array_contains('outlet_type', 'Type1'))

data_with_check.display()

### Group_By

In [0]:
data.display()

In [0]:
## Find MRP for each item type

x = data.groupBy("Item_Type").agg(F.round(F.sum("Item_MRP"),2).alias("total"))

x.display()

In [0]:
y = (
    data.groupBy(["Item_Type","Outlet_Size"])
    .agg(
        F.round(F.avg("Item_MRP"),2)
        .alias("average_MRP_by_type_and_size"),
        F.round(F.sum("Item_MRP"),2)
        .alias("total_MRP_by_type_and_size")
    )
    .sort(
        F.col("Outlet_Size").asc_nulls_first(),
        F.col("average_MRP_by_type_and_size").desc()
    )
)
y.display()